# SCOS Table Corner Detection — Kaggle Training (v2)

Trains a **YOLOv8 keypoint (pose) model** that detects 4 snooker table corners:
`TL → TR → BR → BL`

This replaces manual click-to-calibrate with **automatic AI calibration**.

## v2 Improvements
- **954 frames** (107 original + 847 Albumentations-augmented)
- **flip_idx** fixed to enable horizontal flip augmentation during training
- Multiple camera angles (Table 1, Table 2, Table 7)

## Before running
1. Set Accelerator → **GPU (T4 x2 or P100)**
2. Add your annotated dataset as a Kaggle dataset named `scos-corners-v2`
3. Add `HF_TOKEN` in Kaggle Secrets (Settings → Secrets → Add)
4. Edit Cell 3: set `HF_REPO_ID`

In [ ]:
# Cell 1: Install dependencies
!pip install -q ultralytics huggingface_hub

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2: Check dataset structure (v2 — flat directory)
import os, glob

# After adding your Kaggle dataset, it appears here:
DATASET_PATH = '/kaggle/input/scos-corners-v2'

# v2 dataset has flat structure: images and labels in the same directory
all_jpgs = glob.glob(f'{DATASET_PATH}/*.jpg')
all_txts = glob.glob(f'{DATASET_PATH}/*.txt')
print(f'JPG files: {len(all_jpgs)}')
print(f'TXT files: {len(all_txts)}')

# Sample a label to verify format
if all_txts:
    with open(all_txts[0]) as f:
        content = f.read()
    print(f'\nSample label ({os.path.basename(all_txts[0])}):')
    print(content)

In [ ]:
# Cell 3: Configuration — EDIT THESE

DATASET_PATH   = '/kaggle/input/scos-corners-v2'
HF_REPO_ID     = 'asadahsan148/scos-corner-detector'  # CHANGE THIS
MODEL_SIZE     = 'yolov8s-pose.pt'   # pose model for keypoint detection
EPOCHS         = 150
BATCH_SIZE     = 16
IMGSZ          = 640
TRAIN_SPLIT    = 0.85   # 85% train, 15% val

print(f'Model : {MODEL_SIZE}')
print(f'Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, ImgSz: {IMGSZ}')
print(f'HF Repo: {HF_REPO_ID}')

In [ ]:
# Cell 4: Prepare dataset — flat structure, convert to YOLO pose format + train/val split
import os, shutil, random, glob
import yaml

random.seed(42)
WORK = '/kaggle/working/corner_dataset'

for split in ['train', 'val']:
    os.makedirs(f'{WORK}/{split}/images', exist_ok=True)
    os.makedirs(f'{WORK}/{split}/labels', exist_ok=True)

# v2 dataset: flat directory with .jpg and .txt files together
all_jpgs = sorted(glob.glob(f'{DATASET_PATH}/*.jpg'))
labeled = []
for f in all_jpgs:
    stem = os.path.splitext(os.path.basename(f))[0]
    lbl = f'{DATASET_PATH}/{stem}.txt'
    if os.path.exists(lbl):
        labeled.append((f, lbl))

print(f'Labeled frames: {len(labeled)}')
if len(labeled) < 10:
    print('WARNING: Very few labeled frames. Annotate more for better results!')

random.shuffle(labeled)
split_idx = int(len(labeled) * TRAIN_SPLIT)
train_set = labeled[:split_idx]
val_set   = labeled[split_idx:]
print(f'Train: {len(train_set)}, Val: {len(val_set)}')

# Each label file has 4 lines (one per keypoint): class nx ny
# YOLO pose needs: class cx cy w h  kp1x kp1y v1  kp2x kp2y v2  kp3x kp3y v3  kp4x kp4y v4
# We use the bounding box of all 4 keypoints as the object box.

def convert_point_label_to_pose(label_path):
    """Convert 4-point label (4 rows: class x y) to YOLO pose format (1 row)."""
    pts = []
    with open(label_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 3:
                pts.append((float(parts[1]), float(parts[2])))
    if len(pts) != 4:
        return None

    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    cx = (min(xs) + max(xs)) / 2
    cy = (min(ys) + max(ys)) / 2
    bw = max(xs) - min(xs) + 0.02
    bh = max(ys) - min(ys) + 0.02

    # YOLO pose: class cx cy bw bh  kp1x kp1y 2  kp2x kp2y 2  kp3x kp3y 2  kp4x kp4y 2
    # visibility=2 means visible
    kp_str = '  '.join([f'{px:.6f} {py:.6f} 2' for px, py in pts])
    return f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}  {kp_str}'

def copy_split(split_data, split_name):
    count = 0
    for img_path, lbl_path in split_data:
        pose_line = convert_point_label_to_pose(lbl_path)
        if pose_line is None:
            print(f'  Skipped (bad label): {lbl_path}')
            continue
        stem = os.path.splitext(os.path.basename(img_path))[0]
        shutil.copy(img_path, f'{WORK}/{split_name}/images/{stem}.jpg')
        with open(f'{WORK}/{split_name}/labels/{stem}.txt', 'w') as f:
            f.write(pose_line + '\n')
        count += 1
    print(f'{split_name}: {count} files ready')

copy_split(train_set, 'train')
copy_split(val_set,   'val')

# Write data.yaml for YOLO pose — with flip_idx to enable horizontal flip augmentation
# flip_idx maps keypoints after flip: TL(0)<->TR(1), BR(2)<->BL(3)
data_cfg = {
    'path': WORK,
    'train': 'train/images',
    'val':   'val/images',
    'nc': 1,
    'names': ['table'],
    'kpt_shape': [4, 3],  # 4 keypoints, 3 values each (x, y, visibility)
    'flip_idx': [1, 0, 3, 2],  # TL<->TR, BR<->BL on horizontal flip
}

yaml_path = '/kaggle/working/corners.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_cfg, f)

print(f'\ndata.yaml written: {yaml_path}')
with open(yaml_path) as f:
    print(f.read())

In [ ]:
# Cell 5: Train YOLOv8-pose for corner keypoint detection
from ultralytics import YOLO

model = YOLO(MODEL_SIZE)

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    device=0,
    project='/kaggle/working/scos_corners_train',
    name='corner_detector',
    save=True,
    save_period=25,
    cache=True,
    workers=4,
    patience=40,
    lr0=0.01,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=5,
    close_mosaic=15,
    amp=True,
    augment=True,
    degrees=10.0,   # rotate slightly for augmentation
    scale=0.3,      # scale augmentation
    fliplr=0.5,     # horizontal flip
    mosaic=0.5,
    plots=True,
    verbose=True,
    seed=42,
)

print('Training complete!')

In [ ]:
# Cell 6: Validate — check keypoint accuracy
metrics = model.val()
print(f'Pose mAP@0.5:      {metrics.pose.map50:.4f}')
print(f'Pose mAP@0.5:0.95: {metrics.pose.map:.4f}')
print(f'Box  mAP@0.5:      {metrics.box.map50:.4f}')

In [ ]:
# Cell 7: Visual check — run on a few val images and draw corners
import cv2, glob, os
import numpy as np
from IPython.display import Image as IPyImage, display

# Find the latest training run (Kaggle auto-increments run names like corner_detector-2, -3, etc.)
train_runs = sorted(glob.glob('/kaggle/working/scos_corners_train/corner_detector*/weights/best.pt'))
BEST_PT = train_runs[-1] if train_runs else '/kaggle/working/scos_corners_train/corner_detector/weights/best.pt'
print(f'Using: {BEST_PT}')
model = YOLO(BEST_PT)

CORNER_COLORS = [(0,255,255),(255,0,255),(255,255,0),(0,255,0)]
CORNER_NAMES  = ['TL','TR','BR','BL']

val_imgs = glob.glob(f'{WORK}/val/images/*.jpg')[:4]
for img_path in val_imgs:
    results = model(img_path, verbose=False)
    r = results[0]
    img = cv2.imread(img_path)
    
    if r.keypoints is not None and len(r.keypoints.xy) > 0:
        kps = r.keypoints.xy[0].cpu().numpy().astype(int)
        for i, (x, y) in enumerate(kps):
            cv2.circle(img, (x, y), 10, CORNER_COLORS[i], -1)
            cv2.putText(img, CORNER_NAMES[i], (x+12, y-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, CORNER_COLORS[i], 2)
        # Draw table outline
        pts = kps.reshape(-1, 1, 2)
        cv2.polylines(img, [pts], isClosed=True, color=(0,200,255), thickness=2)
    
    out = f'/kaggle/working/check_{os.path.basename(img_path)}'
    cv2.imwrite(out, img)
    display(IPyImage(out))
    
print('Visual check complete')

In [ ]:
# Cell 8: Export to ONNX
import glob

# Find the latest training run
train_runs = sorted(glob.glob('/kaggle/working/scos_corners_train/corner_detector*/weights/best.pt'))
BEST_PT = train_runs[-1] if train_runs else '/kaggle/working/scos_corners_train/corner_detector/weights/best.pt'
print(f'Using: {BEST_PT}')
model = YOLO(BEST_PT)

onnx_path = model.export(format='onnx', imgsz=IMGSZ, simplify=True, dynamic=False)
print(f'ONNX: {onnx_path}')

In [ ]:
# Cell 9: Push model to Hugging Face Hub
import os, glob
from huggingface_hub import HfApi, create_repo
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
api = HfApi(token=hf_token)

create_repo(HF_REPO_ID, token=hf_token, repo_type='model', exist_ok=True)
print(f'Repo: https://huggingface.co/{HF_REPO_ID}')

# Find the latest training run
train_runs = sorted(glob.glob('/kaggle/working/scos_corners_train/corner_detector*/weights/best.pt'))
WEIGHTS_DIR = os.path.dirname(train_runs[-1]) if train_runs else '/kaggle/working/scos_corners_train/corner_detector/weights'
RUN_DIR = os.path.dirname(WEIGHTS_DIR)
print(f'Weights dir: {WEIGHTS_DIR}')

uploads = [
    ('best.pt',   f'{WEIGHTS_DIR}/best.pt'),
    ('last.pt',   f'{WEIGHTS_DIR}/last.pt'),
    ('best.onnx', onnx_path),
]

for fname, fpath in uploads:
    if os.path.exists(fpath):
        api.upload_file(path_or_fileobj=fpath, path_in_repo=fname,
                        repo_id=HF_REPO_ID, repo_type='model', token=hf_token)
        print(f'  Uploaded: {fname}')

# Upload training plot
results_png = f'{RUN_DIR}/results.png'
if os.path.exists(results_png):
    api.upload_file(path_or_fileobj=results_png, path_in_repo='results.png',
                    repo_id=HF_REPO_ID, repo_type='model', token=hf_token)
    print('  Uploaded: results.png')

print(f'\nDone! Model at: https://huggingface.co/{HF_REPO_ID}')

In [ ]:
# Cell 10: Upload model card (README.md)
model_card = f"""---
license: apache-2.0
task_categories:
- keypoint-detection
- object-detection
tags:
- yolo
- yolov8-pose
- snooker
- table-detection
- calibration
- computer-vision
library_name: ultralytics
---

# SCOS Snooker Table Corner Detector

Detects the **4 corners of a snooker table** (TL, TR, BR, BL) for automatic
perspective calibration in the SCOS (Snooker Club Operating System).

Replaces manual click-to-calibrate with a single model inference call.

## Model Details
- **Architecture**: YOLOv8s-pose (keypoint detection)
- **Keypoints**: 4 (Top-Left, Top-Right, Bottom-Right, Bottom-Left)
- **Image size**: {IMGSZ}x{IMGSZ}
- **Class**: `table` (1 class)

## Usage

```python
from ultralytics import YOLO

model = YOLO('{HF_REPO_ID}')  # auto-download from HF Hub
results = model('frame.jpg')

# Get corners: [TL, TR, BR, BL]
corners = results[0].keypoints.xy[0].cpu().numpy()
# corners[0] = TL, corners[1] = TR, corners[2] = BR, corners[3] = BL
```

## SCOS Integration

The SCOS backend auto-calibration route calls the HF Space inference endpoint,
which runs this model and returns the 4 corner coordinates directly.
These are fed into the existing perspective warp pipeline without any manual input.

## Training Data
Annotated frames extracted from live CCTV footage of snooker tables.
Labels: 4 keypoints per frame in YOLO pose format.
"""

readme_path = '/kaggle/working/README.md'
with open(readme_path, 'w') as f:
    f.write(model_card)

api.upload_file(path_or_fileobj=readme_path, path_in_repo='README.md',
                repo_id=HF_REPO_ID, repo_type='model', token=hf_token)
print('Model card uploaded.')
print(f'\nModel ready at: https://huggingface.co/{HF_REPO_ID}')